# fastcausal Demo

Code to demonstrate basic fastcausal capabilities in a Jupyter notebook.

In [ ]:
from fastcausal import FastCausal

fc = FastCausal()

### Reading in data

For this demo we use a sample EMA dataset bundled with fastcausal.

To read in your own CSV data file `"mydata.csv"` you would use pandas:

```python
import pandas as pd
df = pd.read_csv("mydata.csv")
```

In [ ]:
# Load the sample EMA dataset
df = fc.load_sample("boston")
df

In [ ]:
# Add lagged columns (yesterday's values) with a '_lag' suffix
lag_stub = "_lag"
df_lag = fc.add_lag_columns(df, lag_stub=lag_stub)
df_lag

In [ ]:
# Standardize all columns to zero mean and unit variance
df_std = fc.standardize(df_lag)
df_std

In [ ]:
# Get the original column names
cols = df.columns
cols

In [ ]:
# Create temporal prior knowledge explicitly.
# Tier 0 = lag variables (can only be parents)
# Tier 1 = current-day variables
# Lag variables can cause current-day variables, but not vice versa.

knowledge = {
    "addtemporal": {
        0: [col + lag_stub for col in cols],
        1: [col for col in cols],
    }
}
knowledge

In [ ]:
# Run GFCI causal discovery search
result, graph = fc.run_search(
    df_std,
    algorithm="gfci",
    alpha=0.01,
    penalty_discount=1.0,
    knowledge=knowledge,
)

In [ ]:
fc.show_graph(graph)

In [ ]:
print(f"Edges: {result['num_edges']}")
for e in result["edges"]:
    print(f"  {e}")

Show the graph with just the directed edges.

In [ ]:
fc.show_graph(graph, directed_only=True)

### Node styling

Let's highlight variable groups: lag variables get a dotted outline,
PANAS_PA is light green, PANAS_NA is light pink, and alcohol_bev
variables are purple rectangles.

In [ ]:
node_styles = [
    {"pattern": "*_lag",        "style": "dotted"},
    {"pattern": "PANAS_PA*",    "style": "filled", "fillcolor": "lightgreen"},
    {"pattern": "PANAS_NA*",    "style": "filled", "fillcolor": "lightpink"},
    {"pattern": "PANAS_PA_lag", "style": "filled,dotted", "fillcolor": "lightgreen"},
    {"pattern": "PANAS_NA_lag", "style": "filled,dotted", "fillcolor": "lightpink"},
    {"pattern": "alcohol_bev*", "shape": "box", "style": "filled",
     "fillcolor": "purple", "fontcolor": "white"},
]

fc.show_graph(graph, node_styles=node_styles)

In [ ]:
# Styled graph with directed edges only
fc.show_graph(graph, node_styles=node_styles, directed_only=True)

### Multi-graph comparison

Compare results from different penalty discount values side-by-side.
Nodes are pinned at shared positions so differences are easy to spot.
Disconnected nodes are grayed out by default.

In [ ]:
# Run two more searches with higher penalty discounts
result2, graph2 = fc.run_search(
    df_std, algorithm="gfci", alpha=0.01,
    penalty_discount=2.0, knowledge=knowledge,
)

result3, graph3 = fc.run_search(
    df_std, algorithm="gfci", alpha=0.01,
    penalty_discount=3.0, knowledge=knowledge,
)

In [ ]:
# Compare three graphs side-by-side
fc.show_n_graphs(
    [graph, graph2, graph3],
    node_styles=node_styles,
    gray_disconnected=True,
    labels=["PD=1.0", "PD=2.0", "PD=3.0"],
    graph_size="10,8",
)

In [ ]:
# Directed edges only
fc.show_n_graphs(
    [graph, graph2, graph3],
    node_styles=node_styles,
    gray_disconnected=True,
    directed_only=True,
    labels=["PD=1.0", "PD=2.0", "PD=3.0"],
    graph_size="10,8",
)

In [ ]:
# Without graying out disconnected nodes
fc.show_n_graphs(
    [graph, graph2, graph3],
    node_styles=node_styles,
    gray_disconnected=False,
    labels=["PD=1.0", "PD=2.0", "PD=3.0"],
    graph_size="10,8",
)

In [ ]:
# Save graphs to PNG files
fc.save_n_graphs(
    [graph, graph2, graph3],
    ["paired_graph_pd1", "paired_graph_pd2", "paired_graph_pd3"],
    node_styles=node_styles,
    gray_disconnected=True,
    labels=["PD=1.0", "PD=2.0", "PD=3.0"],
    graph_size="10,8",
    res=300,
)
print("Saved: paired_graph_pd1.png, paired_graph_pd2.png, paired_graph_pd3.png")